# EmpowerLens — Cascade architecture Kaggle GPU runner

Runs the two-stage cascade end to end on Kaggle's free GPU:

```
Input -> Stage 1: binary model (distorted vs not, trained on the FULL splits)
      -> if distorted -> Stage 2: multilabel head (10 types, trained on DISTORTED-ONLY rows)
```

**Before running:**
1. Settings -> **Accelerator: GPU**, and **Internet: On**.
2. `data/splits_combined/` must already be **committed and pushed** on the branch below — this notebook derives Stage 2's splits from it but never regenerates Stage 1's.
3. `src/losses.py`, `src/make_splits_cascade.py`, `src/evaluate_cascade.py`, and the updated `src/train_transformer.py` must be pushed to the branch too — this notebook only orchestrates shell commands, all logic lives in `src/`.

**What this notebook does, in order:**
1. Clone the repo, install deps.
2. Derive `data/splits_stage2/` (distorted-only) from `data/splits_combined/`.
3. Train Stage 1 (binary) on the full combined splits, 3 seeds.
4. Train Stage 2 (multilabel) on the distorted-only splits, 3 seeds, with focal loss + LLRD (mental-roberta) or layer freezing (DeBERTa-v3).
5. Run the end-to-end cascade evaluator for every seed pairing.
6. Compare the cascade's composed test metrics against the existing flat multilabel baseline in `results_combined/`.
7. Copy `results_cascade/` (and the new checkpoints) to Kaggle's output so they're downloadable.

In [ ]:
# 1. Clone the repo and install the transformer stack.
REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "lumia-space"

!rm -rf empowerlens && git clone --branch $BRANCH $REPO_URL empowerlens
%cd empowerlens
!pip install -q -r requirements-transformer.txt
# DeBERTa-v3's tokenizer is SentencePiece-based; make sure it's present.
!pip install -q sentencepiece protobuf

In [ ]:
# 1a. mental/mental-roberta-base is a GATED model on the Hub — you must (1) accept its
#     terms at https://huggingface.co/mental/mental-roberta-base while logged in, then
#     (2) add a Kaggle Secret named HF_TOKEN (Add-ons -> Secrets, top menu) holding a
#     Hugging Face access token with read scope. Without this, training that model 401s.
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Logged in to Hugging Face Hub.")
except Exception as e:
    print(f"[warn] no working HF_TOKEN secret ({e}) — mental/mental-roberta-base will 401 "
          f"until you accept its terms and add the Kaggle secret.")

In [ ]:
# 2. Step 1 — derive Stage 2's distorted-only splits from data/splits_combined.
#    No re-splitting: row-level train/val/test boundaries are inherited unchanged.
COMBINED_SPLITS = "data/splits_combined"
STAGE2_SPLITS   = "data/splits_stage2"

!python -m src.make_splits_cascade --source $COMBINED_SPLITS --out $STAGE2_SPLITS --force

## Step 2 — Stage 1: binary model, 3 seeds

Trained on the FULL `data/splits_combined` (Annotated + CODIPAS train, frozen Annotated val/test) — unchanged from the flat-model runs, since Stage 1 always needed the full label distribution to learn "distorted vs not".

Skip this cell and reuse existing checkpoints if you already trained `binary_*` models on `data/splits_combined` in a prior session — the metric_for_best_model fix only affects the multiclass task, so binary checkpoints from before are still valid.

In [ ]:
SEEDS = (42, 1337, 2024)
STAGE1_MODELS = [
    "mental/mental-roberta-base",
    "microsoft/deberta-v3-base",
]
STAGE1_BATCH_SIZE = {
    "mental/mental-roberta-base": 16,
    "microsoft/deberta-v3-base": 8,   # DeBERTa-v3's disentangled attention OOM'd at 16
}

STAGE1_OUT = "results_stage1"
!mkdir -p $STAGE1_OUT

for model in STAGE1_MODELS:
    tag = model.split("/")[-1]
    bs = STAGE1_BATCH_SIZE.get(model, 16)
    print(f"\n=== Stage 1 binary: {model} (batch_size={bs}) ===")
    for seed in SEEDS:
        ckpt = f"checkpoints/binary_{tag}_{seed}"
        !python -m src.train_transformer --task binary --model $model --seed $seed \
            --device auto --splits $COMBINED_SPLITS --batch-size $bs
        !python -m src.evaluate --checkpoint $ckpt --reference --splits $COMBINED_SPLITS --out $STAGE1_OUT

## Step 3 — Stage 2: multilabel head, distorted-only, 3 seeds

Trained on `data/splits_stage2` — never sees `no_distortion` rows, so its `macro_f1` is the honest "can it tell the 10 types apart" number with no majority-class dilution.

`mental-roberta-base` uses focal loss + layer-wise LR decay (LLRD) — the two new levers for the minority classes (`all_or_nothing`, `mental_filter`, `personalization`).
`deberta-v3-base` uses focal loss + bottom-layer freezing instead of LLRD, since it showed the larger val→test overfitting gap on the flat-model runs.

In [ ]:
STAGE2_OUT = "results_stage2"
!mkdir -p $STAGE2_OUT

# --- mental-roberta-base: focal loss + LLRD -----------------------------------
model = "mental/mental-roberta-base"
tag = model.split("/")[-1]
print(f"\n=== Stage 2 multilabel: {model} (focal + LLRD) ===")
for seed in SEEDS:
    ckpt = f"checkpoints/multilabel_{tag}_{seed}"
    !python -m src.train_transformer --task multilabel --model $model --seed $seed \
        --device auto --splits $STAGE2_SPLITS --batch-size 16 \
        --loss focal --focal-gamma 2.0 \
        --llrd --llrd-decay 0.9 --lr 3e-5 \
        --lr-scheduler cosine --grad-accum 2 \
        --early-stopping-patience 2
    !python -m src.evaluate --checkpoint $ckpt --reference --splits $STAGE2_SPLITS --out $STAGE2_OUT

In [ ]:
# --- deberta-v3-base: focal loss + bottom-layer freezing ----------------------
model = "microsoft/deberta-v3-base"
tag = model.split("/")[-1]
print(f"\n=== Stage 2 multilabel: {model} (focal + freeze-layers) ===")
for seed in SEEDS:
    ckpt = f"checkpoints/multilabel_{tag}_{seed}"
    !python -m src.train_transformer --task multilabel --model $model --seed $seed \
        --device auto --splits $STAGE2_SPLITS --batch-size 8 \
        --loss focal --focal-gamma 2.0 \
        --freeze-layers 6 --dropout 0.25 \
        --lr-scheduler cosine --grad-accum 4 \
        --early-stopping-patience 2
    !python -m src.evaluate --checkpoint $ckpt --reference --splits $STAGE2_SPLITS --out $STAGE2_OUT

## Step 4 — end-to-end cascade evaluation (the number that counts)

`results_stage2/` above scores Stage 2 in isolation on distorted-only inputs — that number looks better than reality because it never sees Stage 1's false negatives. `evaluate_cascade.py` chains Stage 1 -> Stage 2 and scores the **composed** prediction against the FULL val/test set (`data/splits_combined`, which still has its `no_distortion` rows), so any row Stage 1 misses counts as a miss here too.

In [ ]:
CASCADE_OUT = "results_cascade"
!mkdir -p $CASCADE_OUT

STAGE1_TAGS = [m.split("/")[-1] for m in STAGE1_MODELS]

for tag in STAGE1_TAGS:
    print(f"\n=== Cascade eval: {tag} (Stage 1 + Stage 2, same backbone, matched seeds) ===")
    for seed in SEEDS:
        stage1_ckpt = f"checkpoints/binary_{tag}_{seed}"
        stage2_ckpt = f"checkpoints/multilabel_{tag}_{seed}"
        !python -m src.evaluate_cascade \
            --stage1-checkpoint $stage1_ckpt \
            --stage2-checkpoint $stage2_ckpt \
            --splits $COMBINED_SPLITS --out $CASCADE_OUT

## Step 5 — compare against the existing flat multilabel baseline

Bar to beat: the flat mental-roberta-base multilabel model already committed in `results_combined/paper_comparison.csv` (test weighted_f1 = 0.288, macro_f1 = 0.279).

In [ ]:
import pandas as pd
from pathlib import Path

SOURCES = {
    "results_combined (flat)": "results_combined",
    "results_cascade (composed)": CASCADE_OUT,
}

frames = []
for label, folder in SOURCES.items():
    p = Path(folder) / "paper_comparison.csv"
    if p.exists():
        d = pd.read_csv(p)
        d["results_dir"] = label
        frames.append(d)
    else:
        print(f"[skip] {p} not found")

all_results = pd.concat(frames, ignore_index=True)
view = all_results[(all_results["task"] == "multilabel") & (all_results["split"] == "test")]
comparison = (
    view.groupby(["results_dir", "model"])[["weighted_f1", "macro_f1"]]
    .agg(["mean", "std"]).round(3)
)
print(comparison)

all_results.to_csv(f"{CASCADE_OUT}/flat_vs_cascade_comparison.csv", index=False)
print(f"\nWrote comparison table to {CASCADE_OUT}/flat_vs_cascade_comparison.csv")
print("\nIf results_dir=='results_cascade (composed)' beats the flat row on both metrics,")
print("the cascade architecture is validated end-to-end (Stage 1 errors included).")

In [ ]:
# 6. Copy results to the Kaggle output so they can be downloaded from the session.
for folder in (STAGE1_OUT, STAGE2_OUT, CASCADE_OUT):
    !mkdir -p /kaggle/working/$folder
    !cp -r $folder/* /kaggle/working/$folder/
!ls -la /kaggle/working